# Google Cloud Computing (GCP) — 100 Interview Questions & Answers

> GCP is where you **design, deploy, and scale applications** on Google's global cloud infrastructure.

## Table of Contents

- **Section 1: GCP FUNDAMENTALS** (Q1–Q15) — core concepts, resource hierarchy, projects, regions/zones, gcloud CLI basics
- **Section 2: IAM & SECURITY** (Q16–Q28) — roles, policies, service accounts, KMS, security best practices
- **Section 3: COMPUTE SERVICES** (Q29–Q45) — Compute Engine, GKE, App Engine, Cloud Run, Cloud Functions
- **Section 4: STORAGE & DATABASES** (Q46–Q60) — Cloud Storage, Persistent Disk, Cloud SQL, Spanner, Firestore, Bigtable, BigQuery
- **Section 5: NETWORKING** (Q61–Q72) — VPC, subnets, firewall rules, load balancing, CDN, DNS, hybrid connectivity
- **Section 6: BIG DATA, ANALYTICS & ML** (Q73–Q85) — Pub/Sub, Dataflow, Dataproc, Composer, Vertex AI
- **Section 7: DEVOPS, MONITORING & CI/CD** (Q86–Q95) — Cloud Build, Artifact Registry, Terraform, Cloud Monitoring/Logging
- **Section 8: SCENARIO-BASED / ADVANCED** (Q96–Q100) — architecture design, migration, cost optimization, disaster recovery

---
## ============================================================
## **SECTION 1: GCP FUNDAMENTALS (Q1–Q15)** —> core concepts, resource hierarchy, projects, regions/zones, gcloud CLI basics
## ============================================================

### Q1. What is Google Cloud Platform (GCP)?
**Answer:**  
GCP is **Google's public cloud computing platform** offering infrastructure, platform, and
serverless computing services that run on the same infrastructure Google uses internally for
products like Search, Gmail, and YouTube.

- Offers **IaaS, PaaS, and SaaS** models
- Organized into **compute, storage, networking, big data, ML, and management** service categories
- Billed on a **pay-as-you-go** basis with per-second billing for many services
- Competes with AWS and Microsoft Azure as one of the "big three" cloud providers

### Q2. What is the GCP Resource Hierarchy?
**Answer:**  
GCP resources are organized in a **tree-like hierarchy** that enables policy and billing
inheritance:

```
Organization
 └── Folders
      └── Projects
           └── Resources (VMs, buckets, datasets, etc.)
```

- **Organization** — root node, tied to a Google Workspace/Cloud Identity domain
- **Folders** — group projects (e.g., by department or environment)
- **Projects** — the base unit for billing, APIs, and IAM; every resource belongs to exactly one project
- **Resources** — actual services like Compute Engine instances or Cloud Storage buckets
- IAM policies set higher in the hierarchy are **inherited** by everything below

### Q3. What is a GCP Project?
**Answer:**  
A **Project** is the fundamental organizing entity in GCP.

- Has a unique **Project ID**, **Project Name**, and **Project Number**
- Acts as the container for **billing, APIs/services enabled, IAM permissions, and quotas**
- All resources (VMs, buckets, datasets) must belong to a project
- Best practice: use **separate projects** for dev, staging, and production environments

### Q4. What are Regions and Zones in GCP?
**Answer:**  
- A **Region** is an independent geographic area (e.g., `us-central1`, `asia-south1`) containing multiple zones
- A **Zone** is a deployment area within a region (e.g., `us-central1-a`) — think of it as a single data center
- Resources in the **same zone** have low-latency, high-bandwidth networking
- For **high availability**, distribute resources across **multiple zones** within a region, or across **multiple regions** for disaster recovery
- Some resources are **zonal** (VM instances, persistent disks), some are **regional** (regional subnets, regional managed instance groups), and some are **global** (images, snapshots, VPC networks)

In [ ]:
!gcloud compute regions list --limit=5
!gcloud compute zones list --filter="region:us-central1" 

### Q5. What is gcloud, gsutil, and bq?
**Answer:**  
These are the three primary **command-line tools** for interacting with GCP:

- **`gcloud`** — general-purpose CLI for managing most GCP services (Compute, IAM, GKE, networking)
- **`gsutil`** — CLI specifically for **Cloud Storage** operations (upload, download, sync, ACLs)
- **`bq`** — CLI specifically for **BigQuery** operations (queries, datasets, tables, jobs)

All three authenticate via `gcloud auth login` and respect the currently active project/config.

In [ ]:
!gcloud config list
!gcloud config set project my-project-id
!gsutil ls gs://my-bucket/
!bq ls

### Q6. How do you authenticate to GCP?
**Answer:**  
GCP supports multiple authentication mechanisms depending on the use case:

- **User accounts** — `gcloud auth login` for interactive/human access
- **Service Accounts** — machine identities with JSON key files or attached to compute resources
- **Application Default Credentials (ADC)** — `gcloud auth application-default login`, used by client libraries
- **Workload Identity Federation** — lets external identities (AWS, on-prem, GitHub Actions) assume GCP roles **without exporting service-account keys**
- **OAuth 2.0** — for user-consent based access from applications

### Q7. What is the difference between Console, Cloud Shell, and Cloud SDK?
**Answer:**  
- **Cloud Console** — web-based GUI (console.cloud.google.com) for managing resources visually
- **Cloud Shell** — a **free, temporary, browser-based VM** with `gcloud`, `gsutil`, `bq`, Docker, and editors pre-installed; persists 5GB of `$HOME` storage
- **Cloud SDK** — the downloadable package (installed locally) containing `gcloud`, `gsutil`, `bq`, and client libraries for scripting/automation from your own machine or CI/CD pipeline

### Q8. What is a Service Account and how does it differ from a user account?
**Answer:**  
A **Service Account (SA)** is a special type of Google account that represents a **non-human identity** — an application, VM, or service — rather than an individual person.

| | User Account | Service Account |
|---|---|---|
| Represents | A person | An application / workload |
| Auth | Password/OAuth, MFA | Keys, metadata server, impersonation |
| Used by | Humans via Console/gcloud | Code, VMs, pipelines |

- SAs have an email like `name@project-id.iam.gserviceaccount.com`
- Can be **attached to a VM** so code running on it inherits permissions with no key file needed (recommended)
- Should follow **least privilege** — grant only the roles needed

In [ ]:
!gcloud iam service-accounts create my-sa \
    --display-name="My Service Account"
!gcloud projects add-iam-policy-binding my-project-id \
    --member="serviceAccount:my-sa@my-project-id.iam.gserviceaccount.com" \
    --role="roles/storage.objectViewer" 

### Q9. What is a Quota in GCP and why does it matter?
**Answer:**  
**Quotas** are limits GCP imposes on resource usage to **prevent unexpected billing spikes, protect shared infrastructure, and enforce fair usage**.

- **Rate quotas** — e.g., API calls per 100 seconds
- **Allocation quotas** — e.g., max number of vCPUs per region, number of static IPs
- Quotas are set **per project per region** and can be viewed/requested for increase in **IAM & Admin → Quotas**
- Hitting a quota returns a `429` or `RESOURCE_EXHAUSTED` error — important to plan capacity ahead for production workloads

### Q10. What is Cloud Billing and how are budgets/alerts configured?
**Answer:**  
**Cloud Billing** links one or more projects to a **Billing Account** which tracks usage-based charges.

- A **Billing Account** can be linked to multiple projects
- **Budgets & Alerts** let you set a monthly spend threshold and get notified (e.g., at 50%, 90%, 100%) via email or Pub/Sub
- **Billing export** to BigQuery enables detailed cost analysis and dashboards
- **Committed Use Discounts (CUDs)** and **Sustained Use Discounts (SUDs)** reduce compute costs for predictable/long-running workloads

### Q11. What are the different pricing/discount models in GCP Compute?
**Answer:**  
- **On-Demand** — pay per second, no commitment, most expensive
- **Sustained Use Discounts** — automatic discount the longer a VM runs in a month (no action needed)
- **Committed Use Discounts (CUDs)** — 1 or 3-year commitment for up to ~57% discount
- **Preemptible / Spot VMs** — up to 91% cheaper, but can be reclaimed by Google with short notice — ideal for **fault-tolerant, batch, or stateless** workloads
- **Free Tier** — "Always Free" limited usage on select products (e.g., f1-micro instance, 5GB Cloud Storage)

### Q12. What is the difference between IaaS, PaaS, FaaS, and SaaS with GCP examples?
**Answer:**  
| Model | Description | GCP Example |
|---|---|---|
| **IaaS** | Raw infrastructure — VMs, disks, networks | **Compute Engine** |
| **PaaS** | Managed platform, you deploy code | **App Engine, Cloud Run** |
| **FaaS** | Fully serverless, event-driven functions | **Cloud Functions** |
| **SaaS** | Ready-to-use software | **Google Workspace (Gmail, Docs)** |

Moving from IaaS → SaaS trades **control for convenience**: less operational overhead, but less flexibility.

### Q13. What is Cloud Marketplace?
**Answer:**  
**Cloud Marketplace** (formerly GCP Marketplace) is a catalog of **pre-configured, ready-to-deploy software solutions** — both from Google and third-party vendors (Bitnami, Datadog, MongoDB Atlas, etc.).

- Deploy a full **stack (VM + software + config)** in a few clicks
- Billing for marketplace solutions is **consolidated into your GCP invoice**
- Useful for quickly spinning up WordPress, Jenkins, databases, or SaaS integrations without manual setup

### Q14. What is the difference between a Multi-Regional, Regional, and Zonal resource?
**Answer:**  
- **Zonal** — exists in a single zone (e.g., a VM, a zonal persistent disk); fails if that zone goes down
- **Regional** — replicated/managed across multiple zones **within one region** (e.g., regional persistent disks, regional GKE clusters, regional managed instance groups) — survives a zone outage
- **Multi-Regional** — spans **multiple regions** (e.g., multi-region Cloud Storage buckets, BigQuery datasets) — highest availability & latency-optimized for global access, at higher cost

### Q15. How do you organize GCP resources for a large enterprise (multiple teams/environments)?
**Answer:**  
A common enterprise pattern:

- **Organization** node tied to the company domain
- **Folders** per business unit or environment (`Engineering`, `Marketing`, `Prod`, `Non-Prod`)
- **Projects** per application/environment combo (`app1-dev`, `app1-prod`)
- **Shared VPC** so networking is centrally managed while individual teams own their own projects
- **IAM policies** applied at the folder level so they cascade down, with narrower exceptions at the project level
- **Cloud Identity groups** mapped to IAM roles instead of assigning permissions to individual users

---
## ============================================================
## **SECTION 2: IAM & SECURITY (Q16–Q28)** —> roles, policies, service accounts, KMS, security best practices
## ============================================================

### Q16. What is IAM in GCP?
**Answer:**  
**Identity and Access Management (IAM)** controls **who (identity) can do what (role) on which resource**.

- **Member/Principal** — a user, group, service account, or domain
- **Role** — a collection of permissions (e.g., `roles/storage.admin`)
- **Permission** — the smallest unit, e.g., `storage.buckets.list`
- **Policy** — a binding of members to roles on a specific resource
- IAM policies can be attached at the **Organization, Folder, Project, or individual resource** level and are inherited downward

### Q17. What are the three types of IAM roles?
**Answer:**  
- **Basic (Primitive) roles** — `Owner`, `Editor`, `Viewer` — broad, project-wide, **not recommended** for production (too permissive)
- **Predefined roles** — curated by Google per service, e.g., `roles/compute.instanceAdmin`, `roles/bigquery.dataViewer` — the recommended default choice
- **Custom roles** — you define an exact set of permissions when predefined roles are too broad or too narrow

In [ ]:
!gcloud iam roles list --project=my-project-id --limit=5
!gcloud iam roles describe roles/storage.objectViewer

### Q18. What is the Principle of Least Privilege and how do you implement it in GCP?
**Answer:**  
The principle states that **identities should only have the minimum permissions needed** to perform their job.

Implementation in GCP:
- Prefer **predefined/custom roles** over `Owner`/`Editor`
- Grant roles at the **lowest possible level** in the resource hierarchy (resource > project > folder > org)
- Use **IAM Recommender** to detect and remove unused/excessive permissions
- Use **short-lived credentials** (service account impersonation) instead of long-lived keys
- Regularly **audit with Policy Analyzer / Cloud Asset Inventory**

### Q19. What is a Service Account Key and why should you avoid downloading them?
**Answer:**  
A **Service Account Key** is a JSON/P12 credential file that lets code authenticate as that service account.

Risks:
- **Long-lived** — doesn't expire automatically, so a leaked key is a leaked identity, indefinitely
- Easy to accidentally commit to source control or leave on disk

Preferred alternatives:
- **Attach the SA directly to the compute resource** (VM, GKE, Cloud Run) so it authenticates via the metadata server
- **Workload Identity Federation** for external workloads (avoids keys entirely)
- If a key is unavoidable, **rotate frequently** and store in **Secret Manager**, never in code

### Q20. What is Cloud KMS?
**Answer:**  
**Cloud Key Management Service (KMS)** is a managed service for creating, managing, and using
**cryptographic keys** for encryption/decryption and signing.

- Supports **symmetric and asymmetric** keys
- Keys live inside a **Key Ring** within a specific region
- Supports **automatic key rotation**
- Integrates with **Cloud HSM** (hardware security modules) and **External Key Manager (EKM)** for keys stored outside Google
- Used to implement **Customer-Managed Encryption Keys (CMEK)** for services like BigQuery, GCS, and Compute Engine disks

In [ ]:
!gcloud kms keyrings create my-keyring --location=global
!gcloud kms keys create my-key \
    --keyring=my-keyring --location=global \
    --purpose=encryption

### Q21. What is the difference between CMEK and CSEK?
**Answer:**  
- **CMEK (Customer-Managed Encryption Keys)** — you manage the key in **Cloud KMS**, but Google still performs the encryption operations using it
- **CSEK (Customer-Supplied Encryption Keys)** — you supply your **own raw AES-256 key** at request time; Google **never stores it**, so you must keep it safe or the data becomes unrecoverable
- By default, GCP always applies **Google-managed encryption** at rest even if you don't configure CMEK/CSEK

### Q22. What is Secret Manager?
**Answer:**  
**Secret Manager** is a managed service for storing and accessing **API keys, passwords, certificates, and other sensitive config values**.

- Each secret has **versions**, so you can rotate values without changing application code
- Access is controlled via **IAM** (`roles/secretmanager.secretAccessor`)
- Integrates with **Cloud Functions, Cloud Run, GKE, and Compute Engine** to inject secrets at runtime
- All secrets are **encrypted at rest** automatically

In [ ]:
!gcloud secrets create db-password --replication-policy="automatic"
!echo -n "SuperSecret123" | gcloud secrets versions add db-password --data-file=-

### Q23. What is VPC Service Controls?
**Answer:**  
**VPC Service Controls** creates a **security perimeter** around GCP resources (like GCS buckets or BigQuery datasets)
to mitigate **data exfiltration** risk — even if IAM credentials are compromised.

- Prevents access to protected resources from **outside the defined perimeter** (e.g., a stolen service-account key can't be used from an external network)
- Works **in addition to** IAM, not instead of it
- Commonly used in regulated industries (finance, healthcare) requiring strict data boundaries

### Q24. What is Cloud Identity-Aware Proxy (IAP)?
**Answer:**  
**IAP** lets you enforce **access control based on identity and context** for applications running on GCP,
without needing a VPN.

- Sits in front of App Engine, Compute Engine, GKE, or on-prem apps (via connectors)
- Checks the user's **identity + device posture + IP** before allowing the request through
- Enables **BeyondCorp / Zero Trust** style security — "never trust, always verify"
- Commonly used to secure SSH/RDP access to VMs without exposing public IPs

### Q25. What is Security Command Center (SCC)?
**Answer:**  
**SCC** is GCP's centralized **security and risk management dashboard**.

- Provides **asset inventory**, **vulnerability findings**, and **threat detection** across an organization
- Detects misconfigurations (e.g., publicly exposed buckets), anomalous IAM grants, and active threats
- **Standard tier** is free with basic findings; **Premium tier** adds advanced threat detection and compliance monitoring (CIS, PCI-DSS, etc.)

### Q26. What is the difference between Authentication and Authorization in GCP context?
**Answer:**  
- **Authentication** — verifying **who you are** (login, OAuth, service account credentials)
- **Authorization** — verifying **what you're allowed to do** (IAM roles/permissions)

Example: Signing in via `gcloud auth login` is authentication. Whether that user can then delete a Compute Engine VM depends on IAM authorization (does the user have `roles/compute.instanceAdmin`?).

### Q27. What are Organization Policies?
**Answer:**  
**Organization Policy Service** lets admins set **constraints** on how resources can be configured across the
resource hierarchy — a guardrail layer **above IAM**.

Examples:
- `constraints/compute.vmExternalIpAccess` — block VMs from getting public IPs
- `constraints/iam.disableServiceAccountKeyCreation` — prevent creation of SA key files
- `constraints/gcp.resourceLocations` — restrict where resources can be created (data residency)

Unlike IAM (who can do what), Org Policies restrict **what configurations are allowed at all**, regardless of who is trying.

### Q28. How do you audit activity in a GCP project?
**Answer:**  
Use **Cloud Audit Logs**, which automatically capture:

- **Admin Activity logs** — API calls that modify configuration (always on, cannot be disabled)
- **Data Access logs** — reads/writes to user data (off by default for most services due to volume, except BigQuery)
- **System Event logs** — Google-initiated actions (e.g., live migration)
- **Policy Denied logs** — requests denied due to security policy

Logs can be viewed in **Cloud Logging**, exported to **BigQuery/GCS/Pub-Sub** for long-term retention and analysis.

---
## ============================================================
## **SECTION 3: COMPUTE SERVICES (Q29–Q45)** —> Compute Engine, GKE, App Engine, Cloud Run, Cloud Functions
## ============================================================

### Q29. What is Compute Engine?
**Answer:**  
**Compute Engine (GCE)** is GCP's **IaaS virtual machine** service.

- Fully customizable VMs (custom vCPU/RAM via **custom machine types**)
- Supports **Windows, Linux, or custom images**
- Persistent Disks (SSD/HDD) or Local SSDs for storage
- Supports **live migration** during host maintenance (VM doesn't need to reboot)
- Billed per second, with sustained/committed use discounts

In [ ]:
!gcloud compute instances create my-vm \
    --zone=us-central1-a \
    --machine-type=e2-medium \
    --image-family=debian-12 \
    --image-project=debian-cloud

### Q30. What are Managed Instance Groups (MIGs) vs Unmanaged Instance Groups?
**Answer:**  
- **Managed Instance Group (MIG)** — a group of **identical** VMs created from an **instance template**; supports **autoscaling, autohealing, rolling updates**, and load balancing. Recommended for most production workloads.
- **Unmanaged Instance Group** — a group of **dissimilar** VMs manually assembled; used only for legacy load-balancing scenarios where instances differ in configuration
- MIGs can be **zonal** or **regional** (spanning multiple zones for higher availability)

### Q31. What is autoscaling in Compute Engine?
**Answer:**  
**Autoscaling** automatically adds/removes VM instances in a Managed Instance Group based on load, to
maintain performance while controlling cost.

Scaling signals include:
- **CPU utilization**
- **Load balancing serving capacity**
- **Cloud Monitoring custom metrics** (e.g., queue depth)
- **Schedule-based scaling** for predictable traffic patterns

You configure **min/max instance counts** and a target metric threshold.

### Q32. What is Google Kubernetes Engine (GKE)?
**Answer:**  
**GKE** is Google's **managed Kubernetes** service for deploying, scaling, and managing containerized applications.

- **Autopilot mode** — Google manages nodes, sizing, and security config; you only manage workloads (fully hands-off)
- **Standard mode** — you manage node pools, sizes, and configuration for more control
- Integrates with **Cloud Load Balancing, IAM, Cloud Monitoring/Logging, and Artifact Registry**
- Supports **node auto-provisioning**, **cluster autoscaler**, and **multi-cluster/multi-region** deployments

In [ ]:
!gcloud container clusters create-auto my-cluster \
    --region=us-central1
!kubectl get nodes

### Q33. What is the difference between GKE Autopilot and GKE Standard?
**Answer:**  
| | Autopilot | Standard |
|---|---|---|
| Node management | Fully managed by Google | You manage node pools |
| Billing | Per pod resource requests | Per underlying VM |
| Control | Less granular | Full control (node OS, taints, etc.) |
| Ops overhead | Minimal | Higher |

Choose **Autopilot** for simplicity and to reduce ops burden; choose **Standard** when you need fine-grained control over nodes (custom machine types, GPUs/TPUs, DaemonSets, privileged workloads).

### Q34. What is App Engine and what are its two environments?
**Answer:**  
**App Engine** is a fully managed **PaaS** for deploying web apps and APIs without managing servers.

- **Standard Environment** — sandboxed runtime, very fast scale-to-zero, supports specific language runtimes (Python, Java, Node.js, Go, etc.), cheaper for low/spiky traffic
- **Flexible Environment** — runs apps in **Docker containers on Compute Engine VMs**, supports any language/library, but slower scaling and no true scale-to-zero

App Engine automatically handles scaling, versioning, and traffic splitting between versions.

### Q35. What is Cloud Run?
**Answer:**  
**Cloud Run** is a fully managed **serverless platform for running stateless containers**, invoked via HTTP requests or events.

- You provide a **container image**; Cloud Run handles scaling (including **scale-to-zero**), patching, and infrastructure
- Billed **per 100ms of actual request-processing CPU/memory usage**
- Supports **concurrency** (multiple requests per container instance)
- **Cloud Run jobs** support run-to-completion batch tasks (not just HTTP services)
- Ideal middle ground between App Engine (less flexible) and GKE (more ops overhead)

In [ ]:
!gcloud run deploy my-service \
    --image=gcr.io/my-project/my-image \
    --platform=managed \
    --region=us-central1 \
    --allow-unauthenticated

### Q36. What is Cloud Functions?
**Answer:**  
**Cloud Functions** is GCP's **Function-as-a-Service (FaaS)** offering — single-purpose functions triggered by events, without managing containers or servers at all.

Triggers include:
- **HTTP** requests
- **Cloud Storage** events (object created/deleted)
- **Pub/Sub** messages
- **Firestore** document changes

- **Gen 2** functions are built on Cloud Run/Eventarc under the hood, supporting longer timeouts, larger instances, and concurrency
- Best for lightweight, event-driven logic (image thumbnail generation, webhook processors, glue code)

### Q37. How do you choose between Compute Engine, GKE, App Engine, Cloud Run, and Cloud Functions?
**Answer:**  
A rough decision guide, ordered from most control to most abstraction:

1. **Compute Engine** — need full OS control, custom kernels, licensing requirements, legacy lift-and-shift
2. **GKE** — need Kubernetes orchestration, microservices at scale, hybrid/multi-cloud portability
3. **App Engine** — classic monolithic web app, want zero infra management with built-in versioning/traffic-splitting
4. **Cloud Run** — containerized stateless services/APIs, want serverless scaling with more flexibility than Functions
5. **Cloud Functions** — small, single-purpose, event-driven snippets of code

> General rule: prefer the **most managed option that meets your requirements** to minimize operational burden.

### Q38. What are Preemptible VMs and Spot VMs?
**Answer:**  
Both are **short-lived, low-cost** VM types that Google can reclaim with little notice — good for
fault-tolerant, batch, or stateless workloads at up to ~60-91% discount.

- **Preemptible VMs** — legacy version, hard **24-hour maximum** lifetime, fixed discount
- **Spot VMs** — newer version, **no fixed max runtime**, dynamic pricing based on market demand
- Both receive a **30-second termination notice** before reclamation, allowing graceful shutdown
- Not suitable for stateful, latency-sensitive, or long-running critical workloads unless designed to checkpoint/resume

### Q39. What is a Custom Image and a Startup Script in Compute Engine?
**Answer:**  
- **Custom Image** — a snapshot of a fully configured boot disk (OS + installed software + config) that you can reuse to launch new identical VMs quickly, instead of reconfiguring from scratch each time
- **Startup Script** — a script (bash, PowerShell) that runs automatically **every time a VM boots**, useful for installing packages, pulling latest code, or registering with a service — via VM metadata (`startup-script` key)

In [ ]:
!gcloud compute instances create my-vm \
    --zone=us-central1-a \
    --metadata=startup-script='#!/bin/bash
apt-get update && apt-get install -y nginx'

### Q40. What is Live Migration in Compute Engine?
**Answer:**  
**Live Migration** allows Google to move a running VM to another host **without downtime or reboot**, during host maintenance events (hardware failure, software updates, kernel patches).

- Enabled by default for standard VMs (`onHostMaintenance: MIGRATE`)
- **Not available** for GPU-attached instances, preemptible/spot VMs, or sole-tenant nodes with certain configs — these instead use `TERMINATE` and must be manually restarted or handled via MIG autohealing

### Q41. What is a Sole-Tenant Node?
**Answer:**  
A **Sole-Tenant Node** dedicates an entire physical server exclusively to a single customer's VMs, rather than sharing hardware with other GCP customers.

Used for:
- **Licensing requirements** that mandate dedicated hardware (e.g., some Microsoft/Oracle licenses tied to physical cores)
- **Compliance/regulatory** requirements for hardware isolation
- **Node affinity/anti-affinity** rules to control VM placement precisely

### Q42. What is Anthos?
**Answer:**  
**Anthos** is Google's platform for managing applications **consistently across on-premises, GCP, and other clouds (AWS, Azure)**, built on Kubernetes and Istio.

- **Anthos GKE on-prem / bare metal** — run GKE-compatible clusters in your own data center
- **Anthos Config Management** — enforce consistent policies/configs across all clusters via GitOps
- **Anthos Service Mesh** — managed Istio for traffic management, observability, and security between services
- Positioned for **hybrid and multi-cloud** strategies without vendor lock-in to a single cloud's compute layer

### Q43. What is a Managed Instance Group Rolling Update?
**Answer:**  
A **rolling update** replaces VM instances in a MIG gradually with a new template version, minimizing downtime.

Key parameters:
- **Max surge** — how many extra instances can be created above target size during the rollout
- **Max unavailable** — how many instances can be down simultaneously during rollout
- Supports both **proactive** (immediate replacement) and **opportunistic** (only replace when instance is recreated for other reasons) update policies
- Can be **canary tested** by updating a subset of instances first

### Q44. What is the difference between Regional and Global Load Balancers in the context of Compute?
**Answer:**  
- **Regional** load balancers (e.g., Internal TCP/UDP LB) distribute traffic **within a single region** — lower latency for internal, regional-only architectures
- **Global** load balancers (e.g., External HTTP(S) LB) use **Google's global anycast network** to route users to the **closest healthy backend across regions**, enabling true global applications with a single anycast IP
- Global LBs require **backends in a global external configuration**; regional LBs are simpler and cheaper for single-region apps

### Q45. What is Cloud Batch?
**Answer:**  
**Cloud Batch** is GCP's fully managed service for running **batch computing workloads** (HPC, rendering,
scientific simulation) at scale.

- Automatically **provisions and scales Compute Engine VMs** based on job requirements, then tears them down when the job finishes
- Supports **job arrays**, **retry policies**, and can leverage **Spot VMs** for cost savings
- Removes the need to manually manage MIGs/schedulers for batch-style workloads

---
## ============================================================
## **SECTION 4: STORAGE & DATABASES (Q46–Q60)** —> Cloud Storage, Persistent Disk, Cloud SQL, Spanner, Firestore, Bigtable, BigQuery
## ============================================================

### Q46. What is Cloud Storage and what are its storage classes?
**Answer:**  
**Cloud Storage (GCS)** is GCP's **object storage** service for unstructured data (files, backups, media, logs).

Storage classes (all with the **same API**, differing in availability & cost):
- **Standard** — frequently accessed ("hot") data
- **Nearline** — accessed ~once a month, 30-day min storage
- **Coldline** — accessed ~once a quarter, 90-day min storage
- **Archive** — accessed less than once a year, 365-day min storage, cheapest storage but highest retrieval cost

**Object Lifecycle Management** rules can auto-transition or delete objects based on age/conditions.

In [ ]:
!gsutil mb -l us-central1 gs://my-unique-bucket-name/
!gsutil cp localfile.txt gs://my-unique-bucket-name/
!gsutil lifecycle set lifecycle.json gs://my-unique-bucket-name/

### Q47. What is the difference between Multi-Regional, Dual-Region, and Regional Cloud Storage buckets?
**Answer:**  
- **Regional** — data stored in a single region, lowest latency for co-located compute, lowest cost
- **Dual-Region** — data replicated across two specific regions, good balance of availability and geo-redundancy with **turbo replication** option
- **Multi-Regional** — data replicated across a broad geographic area (e.g., all of the US), highest availability, best for globally-accessed content, highest cost

### Q48. What is the difference between Persistent Disk, Local SSD, and Filestore?
**Answer:**  
- **Persistent Disk (PD)** — durable, network-attached block storage for Compute Engine; can be resized live; snapshot-able; available as Standard (HDD), Balanced, SSD, or Extreme
- **Local SSD** — physically attached to the host, extremely low latency/high IOPS, but **ephemeral** — data lost when VM stops (not just terminates)
- **Filestore** — fully managed **NFS file storage** for workloads needing a shared POSIX filesystem across multiple VMs (e.g., render farms, CMS)

### Q49. What is Cloud SQL?
**Answer:**  
**Cloud SQL** is a fully managed **relational database service** supporting **MySQL, PostgreSQL, and SQL Server**.

- Handles **patching, backups, replication, and failover** automatically
- Supports **High Availability (HA)** configuration with a synchronous standby in another zone
- Supports **read replicas** for scaling read-heavy workloads
- Best for traditional relational workloads that **don't need global horizontal scale** — for that, use Spanner instead

In [ ]:
!gcloud sql instances create my-instance \
    --database-version=POSTGRES_15 \
    --tier=db-f1-micro \
    --region=us-central1

### Q50. What is Cloud Spanner?
**Answer:**  
**Cloud Spanner** is Google's globally distributed, **horizontally scalable relational database** offering
**strong consistency** with SQL support — combining the scalability of NoSQL with the consistency of traditional RDBMS.

- Provides **99.999% availability SLA** in multi-region configurations
- Uses **TrueTime** (GPS + atomic clocks) to guarantee external consistency across regions
- Ideal for **global-scale transactional** applications (financial systems, global inventory) needing both horizontal scale and ACID transactions
- Priced by **compute nodes/processing units** + storage, generally more expensive than Cloud SQL

### Q51. What is the difference between Cloud SQL, Cloud Spanner, and BigQuery?
**Answer:**  
| | Cloud SQL | Cloud Spanner | BigQuery |
|---|---|---|---|
| Type | Relational (OLTP) | Distributed relational (OLTP) | Data Warehouse (OLAP) |
| Scale | Single region, vertical + read replicas | Global, horizontal | Petabyte-scale analytics |
| Consistency | Strong (single instance) | Strong (global) | Eventually for streaming inserts |
| Use case | Traditional apps, moderate scale | Global mission-critical apps | Analytics, BI, reporting |

### Q52. What is Firestore?
**Answer:**  
**Firestore** is a fully managed, serverless **NoSQL document database**, the successor to Datastore, built for
mobile/web app development.

- Data organized as **Collections → Documents → Fields** (nested subcollections supported)
- Offers **real-time listeners** — clients get live updates when data changes
- Two modes: **Native mode** (real-time, mobile/web SDKs) and **Datastore mode** (server-side, legacy compatibility)
- Automatic **multi-region replication** with strong consistency

### Q53. What is Cloud Bigtable?
**Answer:**  
**Bigtable** is a fully managed, wide-column **NoSQL database** designed for **massive scale, low-latency** workloads
(the same tech that powers Google Search, Maps, and Gmail internally).

- Ideal for **time-series data, IoT telemetry, financial market data, and large analytical/operational** datasets
- Scales to **petabytes** with consistent single-digit millisecond latency
- Data model: **rows sorted by row key, column families** — schema design (especially row-key choice) is critical for performance
- Integrates natively with **HBase API**, Dataflow, and Dataproc

### Q54. What is Memorystore?
**Answer:**  
**Memorystore** is a fully managed **in-memory data store** compatible with **Redis** and **Memcached** APIs.

- Used for **caching, session storage, real-time leaderboards, and pub/sub messaging patterns**
- Removes the operational burden of patching/scaling/failover for self-managed Redis/Memcached clusters
- Supports **HA (with automatic failover)** for Redis tier

### Q55. What is BigQuery?
**Answer:**  
**BigQuery** is GCP's fully managed, serverless **enterprise data warehouse** for large-scale analytics using
standard SQL.

- **Separates storage and compute** — you pay for storage (cheap) and query processing (per TB scanned, or flat-rate slots) independently
- Can query **petabytes in seconds** using Google's Dremel execution engine
- Supports **federated queries** (querying data directly in GCS, Cloud SQL, Sheets without loading it)
- Built-in **ML (BigQuery ML)**, **geospatial analysis**, and **streaming inserts**

In [ ]:
!bq query --use_legacy_sql=false \
'SELECT name, SUM(number) as total
 FROM `bigquery-public-data.usa_names.usa_1910_2013`
 GROUP BY name ORDER BY total DESC LIMIT 5' 

### Q56. What is the difference between On-Demand and Flat-Rate pricing in BigQuery?
**Answer:**  
- **On-Demand** — pay **per TB of data scanned** by each query; great for unpredictable/sporadic usage
- **Flat-Rate (Capacity-based / Editions)** — purchase a fixed number of **slots** (query processing capacity) for a flat monthly/hourly cost; better for **high, predictable query volume** where on-demand costs would be higher
- BigQuery **editions** (Standard, Enterprise, Enterprise Plus) offer flexible, autoscaling slot commitments as a modern alternative to legacy flat-rate reservations

### Q57. What is Partitioning and Clustering in BigQuery, and why do they matter?
**Answer:**  
- **Partitioning** — divides a table into segments based on a column (commonly a **date/timestamp**, or an integer range); queries that filter on the partition column **scan only relevant partitions**, drastically cutting cost/time
- **Clustering** — sorts data **within each partition** by one or more columns, so BigQuery can skip blocks of data that don't match filter/aggregation predicates

> Combining both is a standard **cost & performance optimization** technique for large tables.

### Q58. What is Cloud Storage Object Versioning and Lifecycle Management?
**Answer:**  
- **Object Versioning** — when enabled on a bucket, overwriting or deleting an object keeps the **previous version** instead of discarding it, protecting against accidental deletion/overwrite
- **Lifecycle Management** — rules that automatically **transition objects to cheaper storage classes** or **delete them** after conditions are met (age, number of newer versions, custom time)

Example lifecycle rule: *"Move objects to Coldline after 90 days, delete after 365 days."*

### Q59. What is Data Transfer options into GCP (Storage Transfer Service, Transfer Appliance)?
**Answer:**  
- **Storage Transfer Service** — managed, scheduled transfers from **on-prem, AWS S3, Azure Blob**, or between GCS buckets, with checksum validation and retry logic
- **Transfer Appliance** — a physical hardware appliance shipped to you for **offline bulk transfer** of very large datasets (100s of TB to PBs) where network transfer would take too long
- **`gsutil rsync`/`cp`** — manual/scripted transfer for smaller, ad hoc datasets
- Choice depends on **data volume** and **available network bandwidth**

### Q60. How do you design a database choice decision tree for a new GCP workload?
**Answer:**  
Ask, in order:

1. **Structured & relational, single region, moderate scale?** → **Cloud SQL**
2. **Structured & relational, need global scale + strong consistency?** → **Cloud Spanner**
3. **Semi-structured/NoSQL, mobile/web app, need real-time sync?** → **Firestore**
4. **NoSQL, massive scale, time-series/IoT, low-latency operational?** → **Bigtable**
5. **Analytics/BI/reporting over large historical datasets?** → **BigQuery**
6. **Need sub-millisecond caching/sessions?** → **Memorystore**

> Real architectures often **combine several** — e.g., Bigtable for raw ingestion, BigQuery for analytics on top.

---
## ============================================================
## **SECTION 5: NETWORKING (Q61–Q72)** —> VPC, subnets, firewall rules, load balancing, CDN, DNS, hybrid connectivity
## ============================================================

### Q61. What is a VPC in GCP?
**Answer:**  
A **Virtual Private Cloud (VPC)** is a **global, virtual network** providing connectivity for your GCP resources.

- **Global resource** — a single VPC can have subnets in **multiple regions**, unlike AWS where VPCs are region-scoped
- Subnets within it are **regional**, and IP ranges don't need to align with physical location
- Supports **firewall rules, routes, VPC peering, Shared VPC, and Private Google Access**
- Every project gets a **default VPC** with pre-created subnets (best practice: disable default and create custom-mode VPCs for production)

In [ ]:
!gcloud compute networks create my-vpc --subnet-mode=custom
!gcloud compute networks subnets create my-subnet \
    --network=my-vpc --region=us-central1 --range=10.0.0.0/24

### Q62. What is the difference between Auto Mode and Custom Mode VPC networks?
**Answer:**  
- **Auto Mode** — automatically creates one subnet per region with predefined IP ranges; simple to start but less control, risk of IP overlap
- **Custom Mode** — you manually define subnets, regions, and IP ranges; **recommended for production** for precise IP planning and avoiding overlaps in hybrid/multi-VPC environments

### Q63. What are Firewall Rules in GCP?
**Answer:**  
Firewall rules control **traffic to and from VM instances**, applied at the **VPC network level** (not per-VM).

- Have a **direction** (ingress/egress), **priority** (0–65535, lower = higher priority), **action** (allow/deny), and **target** (all instances, service account, or network tag)
- **Implied deny-all ingress** and **implied allow-all egress** exist by default at the lowest priority
- Best practice: use **service accounts as targets** rather than network tags for more secure, less spoofable rules
- **Hierarchical firewall policies** can be applied at Org/Folder level to enforce rules across many projects

In [ ]:
!gcloud compute firewall-rules create allow-ssh \
    --network=my-vpc --direction=INGRESS \
    --action=ALLOW --rules=tcp:22 \
    --source-ranges=0.0.0.0/0

### Q64. What is VPC Peering and when would you use it?
**Answer:**  
**VPC Peering** connects two VPC networks (in the same or different projects/organizations) so resources can
communicate using **internal (private) IPs**, without traversing the public internet.

- Traffic stays on **Google's backbone network** — private and low-latency
- **Non-transitive** — if VPC A peers with B, and B peers with C, A **cannot** reach C automatically
- No single point of failure/bandwidth bottleneck (unlike a VPN gateway)
- Use for connecting **separate teams' VPCs**, or SaaS providers connecting to customer VPCs

### Q65. What is Shared VPC?
**Answer:**  
**Shared VPC** allows a **central "host" project** to own the VPC network, while other **"service" projects**
attach to and use its subnets — enabling centralized network administration with decentralized resource ownership.

- Network admins in the host project control **IP ranges, firewall rules, routes**
- Teams in service projects can still create their own VMs/GKE clusters/etc. within the shared subnets, without managing networking themselves
- Common enterprise pattern for **centralizing network security & compliance** across many teams' projects

### Q66. What is Cloud Load Balancing and what types exist?
**Answer:**  
**Cloud Load Balancing** distributes traffic across backend resources for scalability and reliability, fully managed and software-defined (no pre-warming needed, unlike many traditional LBs).

Major types:
- **External HTTP(S) LB (Global)** — Layer 7, anycast IP, for internet-facing web apps
- **External TCP/UDP Network LB (Regional)** — Layer 4, pass-through, high performance
- **Internal HTTP(S) LB (Regional)** — Layer 7 for internal microservices
- **Internal TCP/UDP LB (Regional)** — Layer 4 for internal services
- **SSL Proxy / TCP Proxy LB (Global)** — for non-HTTP(S) TCP/SSL traffic needing a global anycast IP

### Q67. What is Cloud CDN?
**Answer:**  
**Cloud CDN (Content Delivery Network)** caches content at Google's globally distributed edge points-of-presence,
working together with the **External HTTP(S) Load Balancer**.

- Reduces **latency** (content served from the nearest edge to the user) and **origin server load**
- Supports caching for **static content** (images, video, JS/CSS) and even **dynamic content** with proper cache-control headers
- Configurable **cache modes**: `CACHE_ALL_STATIC`, `USE_ORIGIN_HEADERS`, `FORCE_CACHE_ALL`

### Q68. What is Cloud DNS?
**Answer:**  
**Cloud DNS** is a scalable, managed **authoritative DNS service** for publishing domain names and resolving them
via Google's global anycast DNS infrastructure.

- **Public zones** — resolve names for anyone on the internet
- **Private zones** — resolve names **only within specified VPCs**, useful for internal service discovery
- Supports **DNSSEC** for authenticity/integrity, and **DNS forwarding/peering** between private zones

### Q69. What are the options for Hybrid Connectivity (on-prem to GCP)?
**Answer:**  
- **Cloud VPN** — encrypted IPsec tunnel over the public internet; quick to set up, lower cost, up to ~3 Gbps per tunnel (HA VPN provides 99.99% SLA)
- **Cloud Interconnect (Dedicated)** — a direct **physical connection** to Google's network at a colocation facility, 10/100 Gbps circuits, highest bandwidth/lowest latency
- **Cloud Interconnect (Partner)** — connect via a **supported service provider** when a dedicated connection isn't available/needed, offering flexible bandwidth (50 Mbps–50 Gbps)
- **Cross-Cloud Interconnect** — dedicated connectivity directly to **other cloud providers** (AWS, Azure)

Choice depends on required **bandwidth, latency, and budget**.

### Q70. What is Private Google Access and Private Service Connect?
**Answer:**  
- **Private Google Access** — allows VMs with **only internal IPs** (no public IP) to reach Google APIs/services (e.g., Cloud Storage, BigQuery) without traversing the public internet
- **Private Service Connect (PSC)** — lets you access **Google services or third-party/partner services** (or expose your own service to consumers) via a **private endpoint inside your VPC**, avoiding VPC peering complexity and improving isolation

### Q71. What is Network Tiers (Premium vs Standard) in GCP?
**Answer:**  
- **Premium Tier** — traffic travels over **Google's private global backbone network** as much as possible, offering lower latency and higher reliability; default tier, standard pricing
- **Standard Tier** — traffic uses **regular internet routing** (like a typical ISP) for egress, cheaper but with potentially higher latency and no SLA — suitable for cost-sensitive, latency-tolerant workloads within the same region

### Q72. How would you design a secure, multi-tier network architecture in GCP?
**Answer:**  
A typical **3-tier web app** architecture:

- **Web tier** — VMs/GKE behind an **External HTTPS Load Balancer**, in a public subnet, protected by **Cloud Armor** (WAF/DDoS protection)
- **App tier** — internal-only VMs/services (no public IP), reachable via **Internal Load Balancer**, communicating with the web tier over private IPs
- **Data tier** — Cloud SQL/Spanner with **Private Service Connect** or **Private IP**, not exposed publicly
- **Firewall rules** — deny-by-default, allow only required ports between tiers using **service-account-based targeting**
- **VPC Service Controls** around sensitive data stores to prevent exfiltration
- **Cloud NAT** for any outbound internet access needed from private instances (e.g., downloading packages) without giving them public IPs

---
## ============================================================
## **SECTION 6: BIG DATA, ANALYTICS & ML (Q73–Q85)** —> Pub/Sub, Dataflow, Dataproc, Composer, Vertex AI
## ============================================================

### Q73. What is Cloud Pub/Sub?
**Answer:**  
**Pub/Sub** is a fully managed, real-time **messaging service** implementing the publish-subscribe pattern for
building **event-driven and streaming** architectures.

- **Publishers** send messages to a **Topic**; **Subscribers** receive them via a **Subscription**
- Supports **push** (Pub/Sub calls your endpoint) and **pull** (your app polls) delivery
- Guarantees **at-least-once delivery**; supports **ordering keys** for per-key ordered delivery
- Decouples producers and consumers — a core building block for microservices and streaming pipelines

In [ ]:
!gcloud pubsub topics create my-topic
!gcloud pubsub subscriptions create my-sub --topic=my-topic
!gcloud pubsub topics publish my-topic --message="hello world" 

### Q74. What is Cloud Dataflow?
**Answer:**  
**Dataflow** is a fully managed service for executing **batch and streaming data processing pipelines**, built on
the open-source **Apache Beam** SDK.

- Write **one pipeline** (Java/Python/Go) that can run in both **batch and streaming** modes — Beam's unified model
- **Autoscaling** workers up/down based on throughput, with no cluster management required
- Common pattern: **Pub/Sub → Dataflow → BigQuery** for real-time ETL/ELT
- Supports **windowing, triggers, and watermarks** for handling late/out-of-order streaming data

### Q75. What is Cloud Dataproc?
**Answer:**  
**Dataproc** is a fully managed service for running **Apache Hadoop, Spark, Hive, Presto, and Flink** clusters.

- Spins up a cluster in ~90 seconds, and can be **auto-deleted** after a job finishes to save cost
- Good fit when you have **existing Hadoop/Spark code** you want to lift-and-shift to GCP rather than rewrite in Beam/Dataflow
- Supports **ephemeral clusters** (create-run-delete per job) and **autoscaling policies**
- Integrates with GCS as the primary data lake storage layer instead of HDFS

In [ ]:
!gcloud dataproc clusters create my-cluster \
    --region=us-central1 --num-workers=2 \
    --worker-machine-type=e2-standard-2

### Q76. What is the difference between Dataflow and Dataproc?
**Answer:**  
| | Dataflow | Dataproc |
|---|---|---|
| Programming model | Apache Beam (unified batch+stream) | Native Hadoop/Spark/Hive APIs |
| Cluster management | None — fully serverless | You choose cluster size/config |
| Best fit | New pipelines, streaming-first design | Migrating existing Hadoop/Spark jobs |
| Scaling | Automatic | Manual or autoscaling policy |

> Rule of thumb: **new development → Dataflow**; **existing Spark/Hadoop codebase → Dataproc**.

### Q77. What is Cloud Composer?
**Answer:**  
**Cloud Composer** is a fully managed **workflow orchestration service** built on **Apache Airflow**.

- Lets you author pipelines as **DAGs (Directed Acyclic Graphs)** in Python, defining task dependencies and schedules
- Orchestrates jobs **across GCP services** (BigQuery, Dataflow, Dataproc) and even external/on-prem systems
- Handles Airflow's underlying infrastructure (webserver, scheduler, workers, metadata DB) so you just manage DAGs
- Ideal for complex, multi-step **ETL/ELT pipelines with dependencies and retries**

### Q78. What is Vertex AI?
**Answer:**  
**Vertex AI** is Google's **unified machine learning platform**, consolidating the entire ML lifecycle into one
suite of tools.

Key components:
- **Vertex AI Workbench** — managed Jupyter notebooks for experimentation
- **AutoML** — train high-quality models on custom data with minimal ML expertise
- **Custom Training** — bring your own TensorFlow/PyTorch/scikit-learn code, run on managed infrastructure (including GPUs/TPUs)
- **Vertex AI Pipelines** — orchestrate ML workflows (Kubeflow-based)
- **Model Registry & Endpoints** — version, deploy, and serve models with autoscaling
- **Generative AI Studio / Model Garden** — access foundation models (Gemini, PaLM, open models) for GenAI use cases
- **Feature Store, Vertex AI Search, Vector Search** — supporting infra for production ML

### Q79. What is BigQuery ML (BQML)?
**Answer:**  
**BigQuery ML** lets you **create and run machine learning models directly inside BigQuery using SQL** —
no need to export data or write Python.

- Supports linear/logistic regression, k-means clustering, time-series (ARIMA), boosted trees (XGBoost), deep neural nets, and importing **TensorFlow/ONNX** models
- Training happens **where the data already lives**, avoiding costly data movement
- Great for analysts/data engineers who know SQL but not Python, to quickly prototype models on warehouse data

In [ ]:
-- Example BQML syntax (SQL, not Python):
-- CREATE MODEL `mydataset.my_model`
-- OPTIONS(model_type='logistic_reg') AS
-- SELECT * FROM `mydataset.training_data`;

### Q80. What is Dataplex?
**Answer:**  
**Dataplex** is an **intelligent data fabric / data governance** service that lets you centrally
discover, manage, monitor, and govern data across **BigQuery, GCS, and other sources** without moving it.

- Organizes data into **Lakes → Zones → Assets** logically, regardless of physical storage location
- Provides **automated data quality, data lineage, and data discovery/catalog** capabilities
- Helps enforce consistent **access policies and metadata management** across a distributed data estate

### Q81. What is Data Fusion?
**Answer:**  
**Cloud Data Fusion** is a fully managed, **visual (code-free) ETL/ELT tool** based on the open-source
CDAP project.

- **Drag-and-drop UI** for building data pipelines, connecting 150+ pre-built connectors and transformations
- Under the hood, pipelines compile down to **Dataproc/Spark** jobs for execution
- Ideal for teams wanting **low-code data integration** without writing Beam or Spark code directly

### Q82. What is the typical modern data analytics pipeline architecture on GCP?
**Answer:**  
A common end-to-end pattern:

```
Sources → Ingestion → Processing → Storage/Warehouse → BI/ML
```

- **Ingestion**: Pub/Sub (streaming) or Storage Transfer Service/Transfer Appliance (batch)
- **Processing**: Dataflow (stream/batch ETL), or Dataproc for existing Spark jobs
- **Storage/Warehouse**: GCS (data lake, raw/bronze layer) → BigQuery (curated warehouse)
- **Orchestration**: Cloud Composer coordinates dependent jobs and schedules
- **Consumption**: Looker/Looker Studio for BI dashboards, Vertex AI for ML models on top of BigQuery data

### Q83. What is Looker / Looker Studio?
**Answer:**  
- **Looker Studio** (formerly Data Studio) — a **free, easy drag-and-drop dashboard/reporting** tool that connects to BigQuery, Sheets, and 100s of connectors for quick visualizations
- **Looker** — an enterprise **Business Intelligence (BI) and data platform** using **LookML** (a modeling layer) to define consistent business metrics/governance across an organization, with more advanced embedding, governance, and semantic-modeling capabilities than Looker Studio

### Q84. What is the difference between a Data Lake and a Data Warehouse on GCP?
**Answer:**  
- **Data Lake (Cloud Storage)** — stores **raw data in any format** (structured, semi-structured, unstructured) cheaply, schema is applied at read time ("schema-on-read") — flexible but requires more processing before analysis
- **Data Warehouse (BigQuery)** — stores **structured, curated, and modeled data** optimized for fast SQL analytics, schema is defined at write time ("schema-on-write")
- Modern architectures often combine both: **land raw data in GCS (lake)**, then **transform and load curated tables into BigQuery (warehouse)** — sometimes called a "lakehouse" pattern using BigQuery's external tables over GCS.

### Q85. How would you design a real-time streaming analytics pipeline on GCP?
**Answer:**  
Example: real-time IoT sensor monitoring dashboard.

1. **Ingestion** — IoT devices publish readings to a **Pub/Sub topic**
2. **Processing** — a **Dataflow (Apache Beam)** streaming pipeline consumes from Pub/Sub, applies **windowing** (e.g., 1-minute tumbling windows) to aggregate readings, and detects anomalies
3. **Storage** — aggregated results streamed into **BigQuery** for historical analysis, and/or **Bigtable** for high-throughput per-device time-series lookups
4. **Serving** — **Looker Studio** dashboard queries BigQuery for near-real-time visualization; alerting via **Cloud Monitoring** or a Pub/Sub topic feeding Cloud Functions for notifications
5. **Orchestration/monitoring** — Cloud Monitoring dashboards track pipeline lag, error rates, and throughput

---
## ============================================================
## **SECTION 7: DEVOPS, MONITORING & CI/CD (Q86–Q95)** —> Cloud Build, Artifact Registry, Terraform, Cloud Monitoring/Logging
## ============================================================

### Q86. What is Cloud Build?
**Answer:**  
**Cloud Build** is GCP's fully managed **CI/CD service** for building, testing, and deploying code.

- Defined via a `cloudbuild.yaml` with a series of **build steps**, each running in its own container
- Can trigger automatically on **source pushes** (GitHub, Cloud Source Repositories, Bitbucket)
- Common outputs: **Docker images pushed to Artifact Registry**, deployments to **GKE/Cloud Run/App Engine**
- Billed per **build-minute**, with a free tier for limited daily usage

In [ ]:
# Example cloudbuild.yaml steps:
# steps:
# - name: 'gcr.io/cloud-builders/docker'
#   args: ['build', '-t', 'gcr.io/$PROJECT_ID/my-app', '.']
# - name: 'gcr.io/cloud-builders/docker'
#   args: ['push', 'gcr.io/$PROJECT_ID/my-app']

### Q87. What is Artifact Registry?
**Answer:**  
**Artifact Registry** is a fully managed service for storing and managing **build artifacts and dependencies** —
the successor to Container Registry.

- Supports **Docker container images**, plus **Maven, npm, Python (PyPI), Go, and Apt/Yum** package formats
- Regional or multi-regional repositories with **IAM-based access control**
- Integrates with **Cloud Build, GKE, and vulnerability scanning** to catch CVEs in container images automatically

### Q88. What is Infrastructure as Code (IaC) and how is it used with GCP?
**Answer:**  
**IaC** means defining infrastructure (VMs, networks, IAM, etc.) in **declarative config files** that can be
version-controlled, reviewed, and repeatably applied — instead of manual clicking in the Console.

Options on GCP:
- **Terraform** — the most popular, cloud-agnostic tool using the `google` provider; industry standard for GCP IaC
- **Deployment Manager** — Google's native (now legacy-leaning) IaC tool using YAML/Python templates
- **Config Connector** — manage GCP resources **as Kubernetes objects** if you're already GKE/K8s-centric

> Terraform is generally the **recommended default** for most teams today.

In [ ]:
# main.tf (Terraform example)
# resource "google_compute_instance" "vm" {
#   name         = "my-vm"
#   machine_type = "e2-medium"
#   zone         = "us-central1-a"
#   boot_disk {
#     initialize_params { image = "debian-cloud/debian-12" }
#   }
#   network_interface { network = "default" }
# }

### Q89. What is Cloud Deploy?
**Answer:**  
**Cloud Deploy** is a managed **Continuous Delivery** service for automating deployments to **GKE, Cloud Run, and
Anthos** through defined promotion pipelines.

- Models a **delivery pipeline** as a sequence of target environments (e.g., dev → staging → prod)
- Supports **approval gates** before promoting to sensitive environments like production
- Provides built-in **rollback** capability to a previous known-good release
- Complements Cloud Build (which handles the *build/CI* side) by handling the *deployment/CD* side

### Q90. What is Cloud Monitoring and Cloud Logging?
**Answer:**  
Together, these form **Google Cloud's Operations Suite** (formerly Stackdriver):

- **Cloud Monitoring** — collects **metrics** (CPU, latency, custom app metrics), builds **dashboards**, and triggers **alerting policies** based on thresholds
- **Cloud Logging** — collects, stores, analyzes, and lets you search **structured/unstructured logs** from GCP services, VMs (via the Ops Agent), and applications
- Both integrate with **Uptime Checks**, **Error Reporting**, and **Cloud Trace/Profiler** for full observability
- Logs can be routed via **Log Sinks** to BigQuery, GCS, or Pub/Sub for long-term retention or further processing

In [ ]:
!gcloud logging read "resource.type=gce_instance AND severity>=ERROR" --limit=5

### Q91. What is Cloud Trace and Cloud Profiler?
**Answer:**  
- **Cloud Trace** — a **distributed tracing** system that shows **latency data** across the calls in a request as it moves through multiple services, helping pinpoint bottlenecks in microservices architectures
- **Cloud Profiler** — continuously collects **CPU and memory usage profiles** from production applications with minimal overhead, helping identify hot code paths that need optimization

Both aid **performance debugging** in distributed, microservices-based systems that traditional single-server profiling can't easily cover.

### Q92. What is an Alerting Policy in Cloud Monitoring?
**Answer:**  
An **Alerting Policy** defines conditions under which Cloud Monitoring should **notify someone or something**
that a problem may be occurring.

- Based on a **metric threshold** (e.g., CPU > 90% for 5 minutes), **log-based metric**, or **uptime check failure**
- Notification channels include **email, SMS, PagerDuty, Slack, Pub/Sub, and webhooks**
- Supports **multi-condition policies** and **auto-close** after the condition resolves
- Helps implement proactive **SRE-style incident response** instead of relying on user-reported outages

### Q93. What is a Deployment strategy comparison: Rolling Update vs Blue-Green vs Canary?
**Answer:**  
- **Rolling Update** — gradually replaces old instances/pods with new ones; simple, but a bad release can partially affect users during rollout
- **Blue-Green** — run two full identical environments (blue = current, green = new); switch traffic all at once after validation; enables **instant rollback** by switching back, but doubles resource cost temporarily
- **Canary** — release the new version to a **small % of traffic** first, monitor metrics, then gradually increase; minimizes blast radius of a bad release, but requires good monitoring/automation to detect issues early

GCP tools: **MIG rolling updates**, **GKE/Cloud Deploy support all three patterns**, **Traffic Splitting in App Engine/Cloud Run** natively supports canary releases.

### Q94. What is Config Connector?
**Answer:**  
**Config Connector** is a **Kubernetes add-on** that lets you manage GCP resources (Cloud SQL instances, GCS buckets, IAM policies, etc.) **as Kubernetes Custom Resources (CRDs)**, using standard `kubectl apply` / GitOps workflows.

- Useful for teams **already standardized on Kubernetes/GitOps** who want a single control plane for both app and infra resources
- Complements (rather than replaces) Terraform — some teams use both for different resource types

### Q95. How would you set up a full CI/CD pipeline for a containerized app on GCP?
**Answer:**  
A typical end-to-end setup:

1. **Source** — code pushed to a **GitHub/Cloud Source Repositories** branch triggers a webhook
2. **CI (Cloud Build)** — builds the Docker image, runs unit tests, scans for vulnerabilities, and pushes the image to **Artifact Registry**
3. **CD (Cloud Deploy)** — picks up the new image and promotes it through a defined pipeline: **dev → staging → prod**, with a manual approval gate before production
4. **Deployment target** — **GKE** (via `kubectl`/Helm manifests) or **Cloud Run** (revision-based deploy with traffic splitting for canary)
5. **Observability** — Cloud Monitoring dashboards + alerting policies watch error rates/latency post-deploy; automatic **rollback** triggered via Cloud Deploy if health checks fail
6. **IaC** — all of the above infrastructure (clusters, pipelines, IAM) defined in **Terraform**, version-controlled alongside application code

---
## ============================================================
## **SECTION 8: SCENARIO-BASED / ADVANCED (Q96–Q100)** —> architecture design, migration, cost optimization, disaster recovery
## ============================================================

### Q96. How would you migrate an on-premises 3-tier application to GCP with minimal downtime?
**Answer:**  
A phased **"lift-and-shift then optimize"** approach:

1. **Assess** — use **Migration Center** to inventory on-prem servers, dependencies, and right-size target GCP resources
2. **Migrate compute** — use **Migrate to Virtual Machines** to replicate on-prem VMs to Compute Engine with continuous sync, then perform a **low-downtime cutover** once replication is caught up
3. **Migrate database** — use **Database Migration Service (DMS)** for continuous replication from on-prem MySQL/PostgreSQL to Cloud SQL, cutting over once lag is near-zero
4. **Migrate data** — bulk historical data via **Storage Transfer Service** or **Transfer Appliance** for very large volumes
5. **Validate** — run the migrated stack in parallel, compare outputs/performance, then **switch DNS/load balancer traffic**
6. **Optimize** — after stabilizing, **re-architect** components to cloud-native services (e.g., move to GKE/Cloud Run, add autoscaling, CDN) — the "shift" after the "lift" 

### Q97. How do you design a Disaster Recovery (DR) strategy on GCP?
**Answer:**  
Choose a strategy based on acceptable **RTO (Recovery Time Objective)** and **RPO (Recovery Point Objective)**:

- **Backup & Restore** — cheapest, highest RTO/RPO; periodic backups to GCS, restore on demand
- **Pilot Light** — minimal standby infrastructure kept warm (e.g., a small DB replica), scaled up during failover
- **Warm Standby** — a scaled-down but fully functional replica environment running in another region, scaled up on failover
- **Multi-Site Active-Active** — full duplicate environments actively serving traffic in multiple regions simultaneously; lowest RTO/RPO, highest cost/complexity

Supporting GCP tools: **regional/multi-regional storage & databases (Spanner, multi-region GCS)**, **Cloud DNS with health checks for failover routing**, **cross-region Cloud SQL read replicas promoted during failover**.

### Q98. What strategies would you use to optimize costs on GCP?
**Answer:**  
- **Right-size resources** using **Recommender** (identifies over-provisioned VMs)
- Use **Committed Use Discounts** for stable, predictable workloads and **Spot/Preemptible VMs** for fault-tolerant batch jobs
- Set **Budgets & Alerts**, and export billing data to **BigQuery** for detailed cost attribution by label/project
- Apply **Cloud Storage lifecycle rules** to move cold data to Nearline/Coldline/Archive automatically
- Use **BigQuery partitioning/clustering** to reduce bytes scanned per query (directly reduces on-demand cost)
- Prefer **serverless (Cloud Run/Functions)** for spiky workloads to avoid paying for idle capacity
- Regularly **decommission unused resources** (idle IPs, orphaned disks, forgotten dev/test projects)

### Q99. How would you design a globally distributed, highly available web application on GCP?
**Answer:**  
Key architectural elements:

- **Global External HTTPS Load Balancer** with a single anycast IP routes users to the **nearest healthy backend region**
- **Regional Managed Instance Groups** (or **regional GKE clusters**) in multiple regions behind the LB, each with autoscaling and health checks
- **Cloud CDN** enabled at the load balancer to cache static content close to users
- **Cloud Spanner** (or multi-region Cloud SQL with cross-region replicas) for a globally consistent data layer
- **Cloud DNS** with geo-based routing for any components outside the global LB's scope
- **Cloud Armor** in front for DDoS/WAF protection
- **Multi-region Cloud Storage** buckets for static assets and backups
- Automatic **failover**: if a region's health checks fail, the global LB stops routing traffic there without any manual intervention

### Q100. What are common GCP interview scenario questions about troubleshooting a slow application?
**Answer:**  
A structured troubleshooting approach an interviewer expects:

1. **Check Cloud Monitoring dashboards** — CPU/memory/disk/network utilization on the affected VMs or GKE pods
2. **Check Cloud Trace** — identify which downstream service call is contributing most latency in the request chain
3. **Check Cloud Logging** — look for error spikes, slow query logs, or increased retry/timeout patterns
4. **Check the Load Balancer** — backend health check status, 5xx error rates, and whether traffic is unevenly distributed across backends
5. **Check the database** — slow query logs, connection pool exhaustion, replica lag (Cloud SQL/Spanner insights)
6. **Check autoscaling configuration** — is the MIG/GKE cluster scaling up fast enough for the traffic spike, or hitting a **quota limit**?
7. **Correlate with recent deploys** — was a new version rolled out around when the slowdown started? Consider rollback via **Cloud Deploy** while investigating further

> This systematic, "metrics → traces → logs → infra config" funnel is the core of GCP-based observability-driven debugging.

---
## Summary

You've now covered **100 GCP interview questions** spanning:

1. **Fundamentals** — resource hierarchy, projects, regions/zones
2. **IAM & Security** — roles, service accounts, KMS, VPC Service Controls
3. **Compute** — Compute Engine, GKE, App Engine, Cloud Run, Cloud Functions
4. **Storage & Databases** — Cloud Storage, Cloud SQL, Spanner, Firestore, Bigtable, BigQuery
5. **Networking** — VPC, firewall rules, load balancing, hybrid connectivity
6. **Big Data & ML** — Pub/Sub, Dataflow, Dataproc, Composer, Vertex AI
7. **DevOps & Monitoring** — Cloud Build, Terraform, Cloud Monitoring/Logging
8. **Scenario-Based / Advanced** — migration, disaster recovery, cost optimization, troubleshooting

> **Tip:** For interviews, practice explaining trade-offs (e.g., *Cloud SQL vs Spanner*, *Dataflow vs
> Dataproc*, *GKE Autopilot vs Standard*) — interviewers often care more about **reasoning through
> trade-offs** than reciting definitions.
